# Exercise 3.9 — Frame Potential of the Haar Ensemble

**Chapter 3: Haar Ensembles** &nbsp;|&nbsp; *Exercises*

---

This notebook shows that the second frame potential of the Haar ensemble is exactly $F^{(2)}_{\mathrm{Haar}} = 2$, using invariance of the Haar measure to reduce the two-unitary average to the single-unitary fourth moment of the trace. The result is confirmed by Monte Carlo, and the meaning of equality is checked explicitly on two finite ensembles, one that saturates the bound and one that does not.

## 1. Motivation

The frame potential measures how well an ensemble $\mathcal{E}$ of unitaries imitates the Haar measure at a given moment order. For the second moment,

$$F^{(2)}_{\mathcal{E}} = \mathbb{E}_{U, V \sim \mathcal{E}}\, \bigl|\mathrm{Tr}(\hat U^\dagger \hat V)\bigr|^4 .$$

The quantity is built from overlaps between pairs drawn from the ensemble, so it is large when the elements cluster and small when they spread out. It is bounded below by its Haar value, $F^{(2)}_{\mathcal{E}} \geq F^{(2)}_{\mathrm{Haar}}$, and the value of that floor is the point of this exercise. Once the floor is known, the frame potential becomes a practical test: computing a single number for a candidate circuit ensemble decides whether it reproduces Haar statistics at second order, without ever constructing its moment operator.

## 2. Exercise Statement

The second frame potential of an ensemble $\mathcal{E}$ of unitaries is

$$F^{(2)}_{\mathcal{E}} = \mathbb{E}_{U, V \sim \mathcal{E}}\, \bigl|\mathrm{Tr}(\hat U^\dagger \hat V)\bigr|^4 .$$

Show that for the Haar ensemble $F^{(2)}_{\mathrm{Haar}} = 2$. Given that $F^{(2)}_{\mathcal{E}} \geq F^{(2)}_{\mathrm{Haar}}$ for every ensemble, what does equality mean?

## 3. Analytical Derivation

### Step 1: Reduce two unitaries to one

The frame potential is a double average, but the Haar measure makes one of the two integrals trivial. The measure is invariant under inversion, $\hat U \mapsto \hat U^\dagger$, and under right multiplication by any fixed unitary, $\hat U \mapsto \hat U \hat V$. So for any fixed $\hat V$ the product

$$\hat W = \hat U^\dagger \hat V$$

is itself Haar-distributed when $\hat U$ is. The average over $\hat U$ therefore leaves no $\hat V$ dependence at all, and the outer average over $\hat V$ acts on a constant and does nothing:

$$F^{(2)}_{\mathrm{Haar}} = \mathbb{E}_{\hat U, \hat V} \bigl|\mathrm{Tr}(\hat U^\dagger \hat V)\bigr|^4 = \mathbb{E}_{\hat W} \bigl|\mathrm{Tr}\,\hat W\bigr|^4 .$$

### Step 2: Insert the fourth moment of the trace

The remaining average is the $k=2$ Weingarten calculation of Sec. 3.2, in which the two identity-weight terms contribute $D^2$ each and the two swap-weight terms contribute $D$ each:

$$\mathbb{E}_{\hat W} \bigl|\mathrm{Tr}\,\hat W\bigr|^4 = \frac{D^2 + D^2}{D^2 - 1} - \frac{D + D}{D(D^2-1)} = \frac{2D^2 - 2}{D^2 - 1} = 2 .$$

Hence

$$\boxed{F^{(2)}_{\mathrm{Haar}} = 2}$$

exactly, for every $D \geq 2$. The dimension cancels completely, which is what makes the number usable as a fixed target.

### Step 3: What equality means

The frame potential is the squared Hilbert-Schmidt norm of the ensemble's second moment operator

$$\mathcal{M}^{(2)}_{\mathcal{E}}(\cdot) = \mathbb{E}_{\mathcal{E}}\bigl[\hat W^{\otimes 2} (\cdot)\, \hat W^{\dagger \otimes 2}\bigr],$$

and that norm is minimized when the moment operator equals the Haar one. Equality $F^{(2)}_{\mathcal{E}} = 2$ therefore holds precisely when $\mathcal{E}$ reproduces the Haar second moment, which is to say when $\mathcal{E}$ is a unitary 2-design (Chapter 5). An ensemble can be far smaller than the full unitary group and still hit the floor, and the frame potential is blind to everything about it except its second moment.

---
## 4. Symbolic Verification (SymPy)

The reduction to $\mathbb{E}_{\hat W}|\mathrm{Tr}\,\hat W|^4$ is an invariance argument, so the only algebra to check is the Weingarten sum in Step 2.

In [ ]:
import sympy as sp
from sympy import Symbol, Rational, cancel

D = Symbol('D', positive=True, integer=True)

Wg_id = Rational(1) / (D**2 - 1)
Wg_12 = Rational(-1) / (D * (D**2 - 1))

# E|Tr W|^4 = sum over the four (sigma, tau) terms of Wg(mismatch) * (loop factor).
# Matched pairings (sigma = tau) leave two free indices -> D^2; mismatched -> D.
terms = [('id',   'id',   'id',   Wg_id, D**2),
         ('(12)', '(12)', 'id',   Wg_id, D**2),
         ('id',   '(12)', '(12)', Wg_12, D),
         ('(12)', 'id',   '(12)', Wg_12, D)]

total = sp.Integer(0)
for sigma, tau, mismatch, wg, loops in terms:
    contrib = wg * loops
    total += contrib
    print(f'  sigma={sigma:4s} tau={tau:4s} -> mismatch {mismatch:4s} : '
          f'Wg = {str(wg):26s} loops = {str(loops):4s} contrib = {cancel(contrib)}')

F2 = cancel(total)
print()
print(f'E|Tr W|^4 = {F2}')
assert F2 == 2
print('F^(2)_Haar = 2, independent of D.  PASS   (book: 2)')

---
## 5. Numerical Verification

### The invariance step

Step 1 is the only part of the derivation that is not algebra, so it is worth testing on its own. If $\hat W = \hat U^\dagger \hat V$ really is Haar-distributed, then $\mathrm{Tr}(\hat U^\dagger \hat V)$ sampled from independent pairs must have the same law as $\mathrm{Tr}\,\hat W$ sampled from single Haar unitaries. A two-sample Kolmogorov-Smirnov test on $|\mathrm{Tr}|^2$ checks this without assuming the answer.

In [ ]:
import numpy as np
from scipy.stats import unitary_group, ks_2samp

rng = np.random.default_rng(7)
n = 20000

print('Testing that W = U^dag V is Haar-distributed (two-sample KS on |Tr|^2)')
print()
for D_val in [2, 3, 4]:
    U = unitary_group.rvs(D_val, size=n, random_state=rng)
    V = unitary_group.rvs(D_val, size=n, random_state=rng)
    W = unitary_group.rvs(D_val, size=n, random_state=rng)

    tr_pair = np.einsum('nji,nji->n', U.conj(), V)      # Tr(U^dag V)
    tr_sing = np.trace(W, axis1=1, axis2=2)             # Tr(W)

    stat, pval = ks_2samp(np.abs(tr_pair)**2, np.abs(tr_sing)**2)
    print(f'  D={D_val}: KS statistic = {stat:.4f}, p = {pval:.3f}  '
          f'-> {"same law" if pval > 0.01 else "DIFFERENT"}')
    assert pval > 0.01

print()
print('The invariance step is confirmed: the pair overlap has the single-unitary law.  PASS')

### The frame potential itself

In [ ]:
n = 40000
print(f'Monte Carlo frame potential over {n} independent pairs (U, V)')
print()
print(f'{"D":>3} | {"F^(2) (MC)":>12} {"std err":>9} | {"analytic":>9}')
print('-' * 44)

for D_val in [2, 3, 4, 8, 16]:
    U = unitary_group.rvs(D_val, size=n, random_state=rng)
    V = unitary_group.rvs(D_val, size=n, random_state=rng)
    vals = np.abs(np.einsum('nji,nji->n', U.conj(), V))**4
    mc, err = vals.mean(), vals.std() / np.sqrt(n)
    print(f'{D_val:3d} | {mc:12.4f} {err:9.4f} | {2.0:9.4f}')
    assert abs(mc - 2) < 4 * err + 0.05

print()
print('F^(2)_Haar = 2 for every dimension tested.  PASS')

### The bound, and what saturates it

The claim that $F^{(2)}_{\mathcal{E}} = 2$ characterizes 2-designs can be checked exactly on finite ensembles, with no sampling at all. Two single-qubit ensembles make the contrast:

- the **Pauli group** $\{I, X, Y, Z\}$, which is only a 1-design,
- the **Clifford group**, 24 elements up to phase, which is a 2-design (in fact a 3-design at $D=2$).

The identity $F^{(t)}_{\mathcal{E}} = \|\mathcal{M}^{(t)}_{\mathcal{E}}\|_{\mathrm{HS}}^2$ from Step 3 is verified alongside, by building the moment superoperator directly.

In [ ]:
# Single-qubit Pauli group and Clifford group, both exact (no sampling).
I2 = np.eye(2, dtype=complex)
X = np.array([[0, 1], [1, 0]], dtype=complex)
Y = np.array([[0, -1j], [1j, 0]], dtype=complex)
Z = np.diag([1, -1]).astype(complex)
H = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)
S = np.diag([1, 1j]).astype(complex)

pauli_group = [I2, X, Y, Z]

def canonical(M):
    # Strip the global phase so group elements are counted projectively.
    idx = int(np.argmax(np.abs(M) > 1e-9))
    phase = M.flat[idx]
    return tuple(np.round((M * (abs(phase) / phase)).flatten(), 6))

# Generate the Clifford group as words in H and S.
clifford = {canonical(I2): I2}
frontier = [I2]
while frontier:
    new = []
    for M in frontier:
        for g in (H, S):
            N = g @ M
            k = canonical(N)
            if k not in clifford:
                clifford[k] = N
                new.append(N)
    frontier = new
clifford_group = list(clifford.values())
print(f'Pauli group:    {len(pauli_group)} elements')
print(f'Clifford group: {len(clifford_group)} elements (up to global phase)')

In [ ]:
def frame_potential(ensemble, t=2):
    # F^(t) = E_{U,V} |Tr(U^dag V)|^{2t}, averaged over all ordered pairs.
    N = len(ensemble)
    return sum(abs(np.trace(A.conj().T @ B))**(2 * t)
               for A in ensemble for B in ensemble).real / N**2

def moment_superoperator(ensemble, t=2):
    # Matrix of X -> E_W[W^{ot t} X W^{dag ot t}] acting on vec(X).
    dim = ensemble[0].shape[0]**t
    M = np.zeros((dim**2, dim**2), dtype=complex)
    for W in ensemble:
        Wt = W
        for _ in range(t - 1):
            Wt = np.kron(Wt, W)
        M += np.kron(Wt, Wt.conj())
    return M / len(ensemble)

print(f'{"ensemble":>10} | {"F^(2)":>7} | {"||M||_HS^2":>11} | {"rank M":>6} | {"2-design?":>9}')
print('-' * 60)
for name, ens in [('Pauli', pauli_group), ('Clifford', clifford_group)]:
    F2_ens = frame_potential(ens, t=2)
    M = moment_superoperator(ens, t=2)
    hs_norm2 = np.sum(np.abs(M)**2).real
    rank = np.linalg.matrix_rank(M, tol=1e-8)
    is_design = abs(F2_ens - 2) < 1e-9
    print(f'{name:>10} | {F2_ens:7.4f} | {hs_norm2:11.4f} | {rank:6d} | '
          f'{"yes" if is_design else "no":>9}')
    # The frame potential IS the squared HS norm of the moment operator.
    assert abs(F2_ens - hs_norm2) < 1e-9

print()
print('Pauli:    F^(2) = 4 > 2, so the Pauli group is not a 2-design.')
print('Clifford: F^(2) = 2 exactly, saturating the Haar floor.')
print('In both cases F^(2) = ||M^(2)||_HS^2, as claimed in Step 3.  PASS')

### Why the floor is 2

The Haar average projects onto the commutant of $\hat U^{\otimes 2}$, so $\mathcal{M}^{(2)}_{\mathrm{Haar}}$ is an orthogonal projector. For a projector the squared Hilbert-Schmidt norm is just the rank, and by Schur-Weyl duality that rank is the dimension of the commutant, spanned for $D \geq 2$ by the two permutation operators $\{\mathbb{I}, \mathrm{SWAP}\}$. The floor $F^{(2)}_{\mathrm{Haar}} = 2$ is therefore counting permutations, which is why it carries no $D$ dependence.

In [ ]:
M_cliff = moment_superoperator(clifford_group, t=2)

# A 2-design reproduces the Haar second moment, so its moment operator must be
# the Haar projector: Hermitian, idempotent, and of rank equal to |S_2| = 2.
idem_err = np.max(np.abs(M_cliff @ M_cliff - M_cliff))
herm_err = np.max(np.abs(M_cliff - M_cliff.conj().T))
rank = np.linalg.matrix_rank(M_cliff, tol=1e-8)
trace = np.trace(M_cliff).real

print(f'Hermitian error   |M - M^dag|_max = {herm_err:.2e}')
print(f'Idempotency error |M^2 - M|_max   = {idem_err:.2e}')
print(f'rank(M) = {rank}      Tr(M) = {trace:.6f}      ||M||_HS^2 = {np.sum(np.abs(M_cliff)**2).real:.6f}')
assert idem_err < 1e-9 and herm_err < 1e-9 and rank == 2
print()
print('M is an orthogonal projector of rank 2 = dim of the commutant span{I, SWAP},')
print('so ||M||_HS^2 = Tr(M) = rank(M) = 2 = F^(2)_Haar.  PASS')

# The Pauli group fails precisely here: its twirl projects onto a larger algebra.
M_pauli = moment_superoperator(pauli_group, t=2)
print()
print(f'For comparison, rank(M_Pauli) = {np.linalg.matrix_rank(M_pauli, tol=1e-8)} > 2, '
      f'giving F^(2) = {np.sum(np.abs(M_pauli)**2).real:.1f}.')

---
## 6. Physical Interpretation

The whole calculation turns on one fact, that $\hat U^\dagger \hat V$ is Haar-distributed whenever $\hat U$ is. A double average over a group collapses to a single average, and the two-unitary frame potential inherits the value of the one-unitary trace moment. The exercise is really the fourth moment of the trace wearing a different label.

The value 2 counts pairings. In the Weingarten sum the two matched pairings survive with weight $D^2$ and the two mismatched ones subtract with weight $D$, and the ratio conspires so that every power of $D$ cancels. The projector calculation above says the same thing more directly: $F^{(2)}_{\mathrm{Haar}}$ is the rank of the Haar twirl, which is the dimension of the commutant, which by Schur-Weyl duality is the number of permutations of two copies. This is why the answer is $2 = |S_2|$ and not some dimension-dependent expression, and the pattern continues, with $F^{(t)}_{\mathrm{Haar}} = t!$ whenever $D \geq t$ so that the $t!$ permutation operators remain linearly independent.

The bound $F^{(2)}_{\mathcal{E}} \geq 2$ then acquires an operational reading. The frame potential of any ensemble is the squared norm of its second moment operator, and no ensemble can undercut the Haar projector, since the excess is the squared norm of the difference between the two moment operators. Saturation means the difference vanishes, so the ensemble is a unitary 2-design. The Clifford group above makes the point concrete: 24 elements, a vanishing fraction of $U(2)$, and yet its second moment operator is exactly the Haar one. Reproducing low moments is far cheaper than reproducing the measure, and the frame potential is the instrument that detects when it has been achieved.